# Generasi Teks dengan LSTM Tingkat Karakter

**Buku:** *Deep Learning dengan Python* — Bab 14, Deep Learning Generatif

Melatih LSTM untuk memprediksi karakter berikutnya pada tulisan Nietzsche, lalu membangkitkan teks baru dengan *sampling* dengan *temperature*.

> Diadaptasi dari notebook pendamping *Deep Learning with Python* (F. Chollet, Manning; kode berlisensi MIT) dan diperbarui ke **Keras 3** untuk buku *Deep Learning dengan Python: Konsep, Kode, dan Penerapan Modern* (B. S. Negara).

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+, Keras 3. Kode bersifat *backend-agnostik*: dapat dijalankan dengan backend TensorFlow, JAX, atau PyTorch.

**Perubahan utama dari versi asli (Keras 2.0.8):**
- **Bug asli:** `dtype=np.bool` (dihapus sejak NumPy 1.24) → `dtype=bool`.
- `RMSprop(lr=0.01)` → `RMSprop(learning_rate=0.01)`; `keras.Input` ditambahkan.
- Fungsi `sampel()` diberi epsilon dan pengurangan maksimum agar stabil secara numerik.
- Prediksi saat membangkitkan teks memakai `model(x)` (jauh lebih cepat daripada `predict` per karakter).
- Jumlah epoch default 30 (asli 60) dan teks hanya dibangkitkan setiap 5 epoch.

In [ ]:
import os
# Pilih backend: "tensorflow", "jax", atau "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

## 1. Data: tulisan Nietzsche (±600 ribu karakter)

In [ ]:
import numpy as np

jalur = keras.utils.get_file("nietzsche.txt",
                             origin="https://s3.amazonaws.com/text-datasets/nietzsche.txt")
teks = open(jalur, encoding="utf-8").read().lower()
print("Panjang korpus:", len(teks))

## 2. Vektorisasi: potongan 60 karakter → karakter berikutnya

In [ ]:
PANJANG, LANGKAH = 60, 3
kalimat, karakter_berikut = [], []
for i in range(0, len(teks) - PANJANG, LANGKAH):
    kalimat.append(teks[i:i + PANJANG])
    karakter_berikut.append(teks[i + PANJANG])
print("Jumlah sekuens:", len(kalimat))

daftar_karakter = sorted(set(teks))
indeks = {c: i for i, c in enumerate(daftar_karakter)}
print("Karakter unik:", len(daftar_karakter))

x = np.zeros((len(kalimat), PANJANG, len(daftar_karakter)), dtype=bool)   # asli: np.bool
y = np.zeros((len(kalimat), len(daftar_karakter)), dtype=bool)
for i, s in enumerate(kalimat):
    for t, c in enumerate(s):
        x[i, t, indeks[c]] = 1
    y[i, indeks[karakter_berikut[i]]] = 1

## 3. Model

In [ ]:
from keras import layers

model = keras.Sequential([
    keras.Input(shape=(PANJANG, len(daftar_karakter))),
    layers.LSTM(128),
    layers.Dense(len(daftar_karakter), activation="softmax"),
])
model.compile(optimizer=keras.optimizers.RMSprop(learning_rate=0.01),
              loss="categorical_crossentropy")
model.summary()

## 4. *Sampling* dengan temperature
Temperature rendah → teks aman dan berulang; temperature tinggi → teks kreatif tetapi makin acak.

In [ ]:
def sampel(peluang, suhu=1.0):
    peluang = np.asarray(peluang, dtype="float64")
    logit = np.log(peluang + 1e-10) / suhu
    eksp = np.exp(logit - logit.max())
    peluang = eksp / eksp.sum()
    return int(np.argmax(np.random.multinomial(1, peluang, 1)))

def bangkitkan(benih, suhu, n=400):
    hasil = benih
    for _ in range(n):
        x_pred = np.zeros((1, PANJANG, len(daftar_karakter)), dtype="float32")
        for t, c in enumerate(benih):
            x_pred[0, t, indeks[c]] = 1.0
        prediksi = keras.ops.convert_to_numpy(model(x_pred, training=False))[0]
        c_baru = daftar_karakter[sampel(prediksi, suhu)]
        hasil += c_baru
        benih = benih[1:] + c_baru
    return hasil

## 5. Latih dan bangkitkan teks

In [ ]:
import random

EPOCH_TOTAL = 30
TAMPIL_SETIAP = 5
for epoch in range(1, EPOCH_TOTAL + 1):
    print(f"\n=== Epoch {epoch}/{EPOCH_TOTAL} ===")
    model.fit(x, y, batch_size=128, epochs=1)
    if epoch % TAMPIL_SETIAP and epoch != EPOCH_TOTAL:
        continue
    awal = random.randint(0, len(teks) - PANJANG - 1)
    benih = teks[awal:awal + PANJANG]
    print(f'--- Benih: "{benih}"')
    for suhu in [0.2, 0.5, 1.0, 1.2]:
        print(f"\n----- temperature {suhu}:")
        print(bangkitkan(benih, suhu))